In [ ]:
import json
import pandas as pd
import os
import glob

base_path = "AF3_Modeling/Tetramer/Full_Data/"
json_files = glob.glob(os.path.join(base_path, "*.json"))

out_csv_all = os.path.join(base_path, "Figure_5C_contacts_interchain_merged_tetramer.csv")
all_dfs = []

for json_file in json_files:
    filename = os.path.basename(json_file)

    # Extract chain labels from filename
    parts = filename.split("_")
    # fold_YYYY_MM_DD_HH_MM_chainA_chainB(_chainC_chainD...)_full_data_X.json
    chain_labels = []
    for p in parts[6:]:
        if p.lower() == "full":
            break
        chain_labels.append(p)

    # Create chain_map
    chain_ids = [chr(ord("A") + i) for i in range(len(chain_labels))]
    chain_map = dict(zip(chain_ids, chain_labels))

    with open(json_file, "r") as f:
        data = json.load(f)

    if isinstance(data, list):
        if not data:
            continue
        data = data[0]

    contact_probs = data.get("contact_probs", [])
    token_chain_ids = data.get("token_chain_ids", [])
    token_res_ids = data.get("token_res_ids", [])

    if not isinstance(contact_probs, list) or not all(isinstance(row, list) for row in contact_probs):
        raise ValueError(f"'contact_probs' in {filename} should be a list of lists.")
    if len(token_chain_ids) != len(contact_probs) or len(token_res_ids) != len(contact_probs):
        raise ValueError(f"'token_chain_ids' / 'token_res_ids' length mismatch in {filename}.")

    filtered_pairs = []
    for row_idx, row in enumerate(contact_probs):
        for col_idx, prob in enumerate(row):
            if prob is not None and prob > 0.0 and abs(row_idx - col_idx) > 5:
                chain_1 = token_chain_ids[row_idx]
                chain_2 = token_chain_ids[col_idx]

                if chain_1 != chain_2:  # inter-chain
                    mapped_chain1 = chain_map.get(chain_1, chain_1)
                    mapped_chain2 = chain_map.get(chain_2, chain_2)

                    res_1 = token_res_ids[row_idx]
                    res_2 = token_res_ids[col_idx]

                    filtered_pairs.append([res_1, mapped_chain1, res_2, mapped_chain2, prob, filename])

    df = pd.DataFrame(
        filtered_pairs,
        columns=["Residue 1", "Chain 1", "Residue 2", "Chain 2", "Probability", "SourceFile"]
    )
    all_dfs.append(df)

# Merge all
if all_dfs:
    merged_df = pd.concat(all_dfs, ignore_index=True)
    merged_df.to_csv(out_csv_all, index=False)
    print(f"Saved merged file: {out_csv_all}, total pairs: {len(merged_df)}")

    for threshold in [0.2, 0.5, 0.8]:
        subset = merged_df[merged_df["Probability"] >= threshold]
        out_csv_thr = os.path.join(base_path, f"contacts_interchain_merged_tetramer_thr{threshold}.csv")
        subset.to_csv(out_csv_thr, index=False)
        print(f"Saved {out_csv_thr}, pairs: {len(subset)}")
else:
    print("No valid data found.")

In [ ]:
import pandas as pd

# File path
file1 = "Figure_5A_contacts_interchain_merged_thr0.5.csv"
file2 = "Figure_5C_contacts_interchain_merged_tetramer_thr0.5.csv"

# Load CSV
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Exclude columns
exclude_cols = ["Probability", "SourceFile"]

# Campare columns
compare_cols = [col for col in df1.columns if col not in exclude_cols]

# Intersection
intersection = pd.merge(df1[compare_cols], df2[compare_cols])
intersection = intersection.drop_duplicates()
intersection["Category"] = "Intersection"

# Exceptional interactions in File1
only_in_df1 = pd.merge(df1[compare_cols], intersection[compare_cols], 
                       how="outer", indicator=True).query('_merge=="left_only"').drop("_merge", axis=1)
only_in_df1["Category"] = "Only_in_File1"

# Exceptional interactions in File2
only_in_df2 = pd.merge(df2[compare_cols], intersection[compare_cols], 
                       how="outer", indicator=True).query('_merge=="left_only"').drop("_merge", axis=1)
only_in_df2["Category"] = "Only_in_File2"

# Merge results
result = pd.concat([intersection, only_in_df1, only_in_df2], ignore_index=True)

# Save
result.to_csv("Figure_5C_Result_with_Category.csv", index=False)